# Exploratory Data Analysis with SQL
Uses SQLite and the IBM course SQL dataset. The notebook prints the schema first and derives a binary success flag for yearly rates.

In [1]:
import sqlite3
import pandas as pd
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(DATA_URL)
df.columns = [c.strip() for c in df.columns]
print("Columns:", df.columns.tolist())
display(df.head())
conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

def run_query(title, query):
    print("\\n" + title + "\\n" + query)
    result = pd.read_sql_query(query, conn)
    display(result)
    return result

Columns: ['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [2]:
run_query("1. Unique launch sites", "SELECT DISTINCT Launch_Site FROM SPACEXTBL ORDER BY Launch_Site;")
run_query("2. First five CCA launch-site records", "SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;")
run_query("3. Total payload mass", "SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg FROM SPACEXTBL;")
run_query("4. Average payload mass for F9 v1.1", "SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_mass_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%';")
run_query("5. Landing outcomes count", "SELECT Landing_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Landing_Outcome ORDER BY total DESC;")
run_query("6. Payload mass by launch site", "SELECT Launch_Site, SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL GROUP BY Launch_Site ORDER BY total_payload_kg DESC;")

\n1. Unique launch sites\nSELECT DISTINCT Launch_Site FROM SPACEXTBL ORDER BY Launch_Site;


,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


\n2. First five CCA launch-site records\nSELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


\n3. Total payload mass\nSELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg FROM SPACEXTBL;


...

\n5. Landing outcomes count\nSELECT Landing_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Landing_Outcome ORDER BY total DESC;


,Landing_Outcome,total
0,Success,38
1,No attempt,21
2,Success (drone ship),14
3,Success (ground pad),9
4,Failure (drone ship),5
5,Controlled (ocean),5
6,Failure,3
7,Uncontrolled (ocean),2
8,Failure (parachute),2
9,Precluded (drone ship),1


\n6. Payload mass by launch site\nSELECT Launch_Site, SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL GROUP BY Launch_Site ORDER BY total_payload_kg DESC;


,Launch_Site,total_payload_kg
0,CCAFS SLC-40,254037
1,KSC LC-39A,208837
2,VAFB SLC-4E,89730
3,CCAFS LC-40,67363


,Launch_Site,total_payload_kg
0,CCAFS SLC-40,254037
1,KSC LC-39A,208837
2,VAFB SLC-4E,89730
3,CCAFS LC-40,67363


In [3]:
run_query("7. Yearly launch counts", "SELECT strftime('%Y', Date) AS launch_year, COUNT(*) AS launches FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;")
yearly_success_query = "SELECT strftime('%Y', Date) AS launch_year, ROUND(100.0 * AVG(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1.0 ELSE 0.0 END), 2) AS success_rate_percent FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;"
run_query("8. Yearly landing success rate", yearly_success_query)
run_query("9. Successful drone-ship landings with payload 4000–6000 kg", "SELECT Booster_Version, PAYLOAD_MASS__KG_, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;")
run_query("10. Maximum payload records", "SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);")
run_query("11. 2015 landing failures", "SELECT Date, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Date LIKE '2015%' AND Landing_Outcome LIKE '%Failure%';")


\n7. Yearly launch counts\nSELECT strftime('%Y', Date) AS launch_year, COUNT(*) AS launches FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;


,launch_year,launches
0,2010,2
1,2012,2
2,2013,3
3,2014,6
4,2015,7
5,2016,8
6,2017,18
7,2018,20
8,2019,11
9,2020,24


\n8. Yearly landing success rate\nSELECT strftime('%Y', Date) AS launch_year, ROUND(100.0 * AVG(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1.0 ELSE 0.0 END), 2) AS success_rate_percent FROM SPACEXTBL GROUP BY launch_year ORDER BY launch_year;


,launch_year,success_rate_percent
0,2010,0.00
1,2012,0.00
2,2013,0.00
3,2014,0.00
4,2015,14.29
5,2016,62.50
6,2017,77.78
7,2018,50.00
8,2019,90.91
9,2020,87.50


\n9. Successful drone-ship landings with payload 4000–6000 kg\nSELECT Booster_Version, PAYLOAD_MASS__KG_, Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;


...

\n10. Maximum payload records\nSELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);


,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


\n11. 2015 landing failures\nSELECT Date, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Date LIKE '2015%' AND Landing_Outcome LIKE '%Failure%';


,Date,Landing_Outcome,Booster_Version,Launch_Site
0,2015-01-10,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,2015-04-14,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


,Date,Landing_Outcome,Booster_Version,Launch_Site
0,2015-01-10,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,2015-04-14,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Notes
Run the queries and use the displayed outputs in your report. If the hosted course CSV schema changes, inspect the printed column list and adjust SQL field names accordingly. Do not paste guessed counts or success percentages.